# 多维输入逻辑回归模型详解
## 一、模型核心原理
### 1.1 模型定位
多维输入逻辑回归是**单变量逻辑回归的自然扩展**，用于处理**多特征输入的二分类任务**（如糖尿病预测：8个特征预测是否患病）。它依然保持“线性组合 + Sigmoid激活”的核心结构，只是将单特征输入扩展为多维特征向量。

### 1.2 数学公式解析
#### 1.2.1 向量形式（简洁表达）
$$
\hat{y}^{(i)} = \sigma\left( x^{(i)} * \omega + b \right)
$$
- $x^{(i)}$：第 $i$ 个样本的**特征向量**（维度为 $d$，如糖尿病数据 $d=8$），即 $x^{(i)} = [x_1^{(i)}, x_2^{(i)}, \dots, x_8^{(i)}]$
- $\omega$：**权重向量**（维度与特征数一致），即 $\omega = [\omega_1, \omega_2, \dots, \omega_8]$
- $b$：偏置项（标量）
- $*$：向量点积（即 $\sum_{n=1}^d x_n^{(i)} \cdot \omega_n$）
- $\sigma(\cdot)$：Sigmoid激活函数，将线性输出映射到 $[0,1]$ 区间

#### 1.2.2 展开形式（显式求和）
$$
\hat{y}^{(i)} = \sigma\left( \sum_{n=1}^8 x_n^{(i)} \cdot \omega_n + b \right)
$$
- 明确展示了**每个特征与对应权重相乘后求和**，再加上偏置项，最后通过Sigmoid得到预测概率
- 以糖尿病数据为例：$n=1$ 到 $8$ 对应 $X_1$ 到 $X_8$ 共8个特征，每个特征都有独立权重 $\omega_n$

### 1.3 与单维逻辑回归的区别
| 维度 | 单维逻辑回归 | 多维逻辑回归 |
|------|--------------|--------------|
| 输入形状 | $(N, 1)$ | $(N, d)$（$d$ 为特征数） |
| 权重形状 | $(1, 1)$ | $(d, 1)$ |
| 线性计算 | $z = x \cdot \omega + b$ | $z = \sum_{n=1}^d x_n \cdot \omega_n + b$ |
| 核心差异 | 仅学习1个特征的权重 | 学习**所有特征的权重**，自动衡量各特征对结果的影响 |

---

## 二、模型计算流程
### 2.1 前向传播步骤
以糖尿病数据（8个特征）为例，单个样本 $x^{(i)}$ 的计算流程：
1. **特征加权求和**：
   $$
   z^{(i)} = x_1^{(i)}\omega_1 + x_2^{(i)}\omega_2 + \dots + x_8^{(i)}\omega_8 + b
   $$
2. **Sigmoid激活**：
   $$
   \hat{y}^{(i)} = \sigma(z^{(i)}) = \frac{1}{1 + e^{-z^{(i)}}}
   $$
3. **类别判定**：
   - 若 $\hat{y}^{(i)} \ge 0.5$，预测为正类（患病，$Y=1$）
   - 若 $\hat{y}^{(i)} < 0.5$，预测为负类（健康，$Y=0$）

### 2.2 批量计算（向量化实现）
为提升效率，实际中会对**批量样本**做向量化计算：
- 输入矩阵 $X$：形状 $(N, d)$（$N$ 为样本数，$d$ 为特征数）
- 权重矩阵 $\omega$：形状 $(d, 1)$
- 偏置 $b$：标量（可广播）
- 批量线性输出：
  $$
  Z = X \cdot \omega + b \quad \text{（形状：}(N,1)\text{）}
  $$
- 批量预测概率：
  $$
  \hat{Y} = \sigma(Z) \quad \text{（形状：}(N,1)\text{）}
  $$

---

## 三、损失函数与优化
### 3.1 二元交叉熵损失（批量形式）
$$
\mathcal{L} = -\frac{1}{N} \sum_{i=1}^N \left[ y^{(i)} \log \hat{y}^{(i)} + (1-y^{(i)}) \log(1-\hat{y}^{(i)}) \right]
$$
- $N$：样本总数
- $y^{(i)}$：第 $i$ 个样本的真实标签（0或1）
- $\hat{y}^{(i)}$：模型预测该样本为正类的概率

### 3.2 梯度计算（核心）
对权重 $\omega_n$ 和偏置 $b$ 求梯度：
1. **对权重 $\omega_n$ 的梯度**：
   $$
   \frac{\partial \mathcal{L}}{\partial \omega_n} = \frac{1}{N} \sum_{i=1}^N \left( \hat{y}^{(i)} - y^{(i)} \right) \cdot x_n^{(i)}
   $$
   直观含义：**预测误差 $(\hat{y}^{(i)}-y^{(i)})$ 乘以对应特征 $x_n^{(i)}$，再对所有样本求平均**
2. **对偏置 $b$ 的梯度**：
   $$
   \frac{\partial \mathcal{L}}{\partial b} = \frac{1}{N} \sum_{i=1}^N \left( \hat{y}^{(i)} - y^{(i)} \right)
   $$
   直观含义：**所有样本预测误差的平均值**

### 3.3 参数更新（梯度下降）
$$
\omega_n = \omega_n - \eta \cdot \frac{\partial \mathcal{L}}{\partial \omega_n} \quad (n=1,2,\dots,8)
$$
$$
b = b - \eta \cdot \frac{\partial \mathcal{L}}{\partial b}
$$
- $\eta$：学习率（超参数，控制参数更新步长）
- 多维场景下，**每个特征的权重都会独立更新**，模型会自动学习不同特征的重要性

---

## 四、PyTorch 多维逻辑回归实现（以糖尿病数据为例）
### 4.1 完整代码


In [1]:
import torch
import numpy as np

xy = np.loadtxt('data/diabetes.csv', delimiter=',', dtype=np.float32)
x_data = torch.from_numpy(xy[:, :-1])
y_data = torch.from_numpy(xy[:, [-1]])
print('x_data',x_data)
print('y_data',y_data)


x_data tensor([[-0.2900,  0.4900,  0.1800, -0.2900,  0.0000,  0.0000, -0.5300, -0.0300],
        [-0.8800, -0.1500,  0.0800, -0.4100,  0.0000, -0.2100, -0.7700, -0.6700],
        [-0.0600,  0.8400,  0.0500,  0.0000,  0.0000, -0.3100, -0.4900, -0.6300],
        [-0.8800, -0.1100,  0.0800, -0.5400, -0.7800, -0.1600, -0.9200,  0.0000],
        [ 0.0000,  0.3800, -0.3400, -0.2900, -0.6000,  0.2800,  0.8900, -0.6000],
        [-0.4100,  0.1700,  0.2100,  0.0000,  0.0000, -0.2400, -0.8900, -0.7000],
        [-0.6500, -0.2200, -0.1800, -0.3500, -0.7900, -0.0800, -0.8500, -0.8300],
        [ 0.1800,  0.1600,  0.0000,  0.0000,  0.0000,  0.0500, -0.9500, -0.7300],
        [-0.7600,  0.9800,  0.1500, -0.0900,  0.2800, -0.0900, -0.9300,  0.0700],
        [-0.0600,  0.2600,  0.5700,  0.0000,  0.0000,  0.0000, -0.8700,  0.1000]])
y_data tensor([[0.],
        [1.],
        [0.],
        [1.],
        [0.],
        [1.],
        [0.],
        [1.],
        [0.],
        [0.]])


In [5]:
class MultipleDimensionInputModel(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.linear1 = torch.nn.Linear(8, 6)
        self.linear2 = torch.nn.Linear(6, 4)
        self.linear3 = torch.nn.Linear(4, 1)
        self.sigmoid = torch.nn.Sigmoid()
    def forward(self, x):
        x = self.sigmoid(self.linear1(x))
        x = self.sigmoid(self.linear2(x))
        x = self.sigmoid(self.linear3(x))
        return x
model = MultipleDimensionInputModel()
criterion = torch.nn.BCELoss(reduction='sum')
optimizer = torch.optim.SGD(model.parameters(), lr=0.05)

for epoch in range(1000):
    # 前向传播
    y_pred = model(x_data)
    loss = criterion(y_pred, y_data)
    # 反向传播
    optimizer.zero_grad()
    loss.backward()
    # 更新参数
    optimizer.step()
    if epoch % 100 == 99:
        print(f"Epoch {epoch}: loss={loss.item():.4f}")

Epoch 99: loss=6.7168
Epoch 199: loss=6.6939
Epoch 299: loss=6.6001
Epoch 399: loss=6.0089
Epoch 499: loss=4.0220
Epoch 599: loss=3.1146
Epoch 699: loss=2.8592
Epoch 799: loss=2.7477
Epoch 899: loss=2.6849
Epoch 999: loss=2.6443
